# Entropy per token (prompt-aligned)

Generate argmax continuations and compute per-token entropy using the same prompt loading and token iteration logic as `pipeline.py`.

In [6]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import sys
import torch
sys.path.append("../")
from plan_trace.utils import load_model, cleanup_cuda


def build_prompt(
    entry: Dict[str, Any],
    *,
    use_nodocstring_prompt: bool = False,
    use_augmented_docstring: bool = False,
) -> str:
    prompt = (
        "You are an expert Python programmer, and here is your task: "
        f"{entry['prompt']} Your code should pass these tests:\n\n"
        + "\n".join(entry["test_list"])
        + "\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\n"
    )
    if use_nodocstring_prompt:
        prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{entry['prompt']} Your code should pass these tests:\n\n"
            + "\n".join(entry["test_list"])
            + "\nWrite your code, without docstrings, below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    if use_augmented_docstring:
        if "augmented_prompt" not in entry:
            raise ValueError("No augmented prompt found in data entry for use_augmented_docstring=True")
        full_augmented_prompt = entry["augmented_prompt"]
        match = re.search(r'"""(.*?)"""', full_augmented_prompt, re.DOTALL)
        if match:
            docstring = match.group(1).strip()
        else:
            raise ValueError("No docstring found in augmented prompt")
        prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{entry['prompt']} {docstring} Your code should pass these tests:\n\n"
            + "\n".join(entry["test_list"])
            + "\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    return prompt


def detect_docstring_tokens(
    model,
    tokens: torch.Tensor,
    start_search_idx: int = 0,
) -> List[Tuple[int, int]]:
    text = model.to_string(tokens[0])
    docstring_ranges: List[Tuple[int, int]] = []
    triple_quotes = [chr(34) * 3, chr(39) * 3]

    for quote in triple_quotes:
        start = 0
        while True:
            start_pos = text.find(quote, start)
            if start_pos == -1:
                break
            end_pos = text.find(quote, start_pos + 3)
            if end_pos == -1:
                break
            start_tokens = len(model.to_tokens(text[:start_pos])[0])
            end_tokens = len(model.to_tokens(text[:end_pos + 3])[0])
            if start_tokens >= start_search_idx:
                docstring_ranges.append((start_tokens, end_tokens))
            start = end_pos + 3
    return docstring_ranges


def generate_argmax_sequence(
    model,
    prompt: str,
    *,
    device: str,
    stop_token_id: int,
    max_new_tokens: int = 150,
) -> torch.Tensor:
    toks_BL = model.to_tokens(prompt).to(device)
    out_BL = toks_BL.clone()

    while out_BL.shape[-1] - toks_BL.shape[-1] < max_new_tokens:
        with torch.no_grad():
            logits_V = model(out_BL)[0, -1]
        next_id = logits_V.argmax(-1).item()
        del logits_V
        cleanup_cuda()
        if next_id == stop_token_id:
            break
        out_BL = torch.cat([out_BL, torch.tensor([[next_id]], device=device)], dim=1)
    return out_BL


def select_top_tokens(
    probs: torch.Tensor,
    *,
    top_p_mass: float = 0.9,
    max_tokens: Optional[int] = None,
) -> List[Tuple[int, float]]:
    values, indices = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(values, dim=0)
    cutoff = (cumulative <= top_p_mass).nonzero(as_tuple=False)
    if cutoff.numel() == 0:
        keep = 1
    else:
        keep = int(cutoff[-1].item()) + 1
    if max_tokens is not None:
        keep = min(keep, max_tokens)
    top_indices = indices[:keep].tolist()
    top_values = values[:keep].tolist()
    return list(zip(top_indices, top_values))


def entropy_from_probs(probs: torch.Tensor) -> float:
    probs = torch.clamp(probs, min=1e-12)
    return float((-probs * torch.log(probs)).sum().item())


def save_entropy_results(results: Dict[str, Any], output_dir: str) -> Path:
    prompt_idx = results.get("prompt_idx", -1)
    folder = Path(output_dir) / f"prompt_{prompt_idx}"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "entropy_token_results.json"
    with open(path, "w") as f:
        json.dump(results, f, indent=2)
    return path


def run_entropy_token_pipeline(
    prompt_idx: int,
    *,
    model_name: str = "gemma-2-2b-it",
    device: str = "cuda",
    use_custom_cache: bool = False,
    skip_docstrings: bool = True,
    start_token_offset: int = 0,
    max_tokens_to_analyze: int = 50,
    stop_token_id: int = 1917,
    data_path: str = "data/first_100_passing_examples.json",
    save_outputs: bool = False,
    output_dir: str = "outputs",
    verbose: bool = True,
    use_augmented_docstring: bool = False,
    save_augmented_docstring: bool = False,
    use_nodocstring_prompt: bool = False,
    use_pos_info: bool = False,
    pos_info_offset: int = 2,
    top_p_mass: float = 0.9,
    top_max_tokens: Optional[int] = 200,
    save_full_probs: bool = False,
) -> Dict[str, Any]:
    if verbose:
        print(f"Starting entropy pipeline for prompt {prompt_idx}")

    model = load_model(model_name, device=device, use_custom_cache=use_custom_cache, dtype=torch.bfloat16)

    with open(data_path, "r") as f:
        data = json.load(f)

    entry = data[prompt_idx]
    prompt = build_prompt(
        entry,
        use_nodocstring_prompt=use_nodocstring_prompt,
        use_augmented_docstring=use_augmented_docstring,
    )

    if verbose:
        print("Generating full sequence...")
    out_BL = generate_argmax_sequence(
        model,
        prompt,
        device=device,
        stop_token_id=stop_token_id,
        max_new_tokens=150,
    )

    if verbose:
        full_response = model.to_string(out_BL[0])
        print("Full generated response:", full_response)
        print(f"Generated sequence length: {out_BL.shape[-1]} tokens")

    prompt_len = model.to_tokens(prompt).shape[-1]

    docstring_ranges: List[Tuple[int, int]] = []
    last_docstring_end: Optional[int] = None
    if not use_nodocstring_prompt and (skip_docstrings or save_augmented_docstring):
        if verbose:
            print("Detecting docstring ranges...")
        docstring_ranges = detect_docstring_tokens(model, out_BL, prompt_len)
        if docstring_ranges:
            last_docstring_end = max(end for _, end in docstring_ranges)

    if skip_docstrings and last_docstring_end is not None:
        first_non_docstring = last_docstring_end + 1
        anchor = max(prompt_len, first_non_docstring)
        start_analysis = anchor + start_token_offset
    else:
        start_analysis = prompt_len + start_token_offset

    end_analysis = min(out_BL.shape[-1] - 1, start_analysis + max_tokens_to_analyze)

    if use_pos_info:
        pos_info = entry.get("position_info", None)
        if pos_info is not None:
            if model_name == "gemma-2-2b-it":
                diff_pos = pos_info.get("instruct_token_pos", None)
            else:
                diff_pos = pos_info.get("base_token_pos", None)
            if diff_pos is not None:
                start_analysis = diff_pos - pos_info_offset
                end_analysis = min(diff_pos + pos_info_offset + 1, out_BL.shape[-1] - 1)

    if verbose:
        print(f"Analyzing token positions {start_analysis} to {end_analysis}")

    with torch.no_grad():
        logits = model(out_BL)  # [1, L, V]

    results: Dict[str, Any] = {
        "prompt_idx": prompt_idx,
        "prompt_length": int(prompt_len),
        "total_length": int(out_BL.shape[-1]),
        "analyzed_range": (int(start_analysis), int(end_analysis)),
        "docstring_ranges": docstring_ranges,
        "top_p_mass": top_p_mass,
        "top_max_tokens": top_max_tokens,
        "token_results": {},
    }

    if save_full_probs:
        results["save_full_probs"] = True

    for token_idx in range(start_analysis, end_analysis):
        logits_V = logits[0, token_idx - 1]
        probs = torch.softmax(logits_V, dim=-1)
        entropy = entropy_from_probs(probs)

        top_tokens = select_top_tokens(probs, top_p_mass=top_p_mass, max_tokens=top_max_tokens)
        top_token_ids = [tok_id for tok_id, _ in top_tokens]
        top_token_strs = model.to_str_tokens(torch.tensor(top_token_ids))
        top_token_entries = [
            {"token_id": tok_id, "token_str": tok_str, "prob": prob}
            for (tok_id, prob), tok_str in zip(top_tokens, top_token_strs)
        ]

        predicted_token_id = int(out_BL[0, token_idx].item())
        token_result: Dict[str, Any] = {
            "prompt_idx": prompt_idx,
            "token_id": int(token_idx),
            "predicted_token_id": predicted_token_id,
            "predicted_token_str": model.to_str_tokens(torch.tensor([predicted_token_id]))[0],
            "entropy": entropy,
            "top_tokens": top_token_entries,
        }

        if save_full_probs:
            token_result["full_probs"] = probs.detach().cpu().tolist()

        results["token_results"][int(token_idx)] = token_result

    if save_augmented_docstring and last_docstring_end is not None:
        if docstring_ranges:
            first_non_docstring = last_docstring_end + 1
            augmented_tokens = out_BL[:, :first_non_docstring]
            augmented_text = model.to_string(augmented_tokens[0])
            if isinstance(augmented_text, str):
                for token in ["<bos>", "<eos>", "<pad>", "<unk>", "<s>", "</s>"]:
                    augmented_text = augmented_text.replace(token, "")
                augmented_text = augmented_text.strip()
            entry["augmented_prompt"] = augmented_text
            data[prompt_idx] = entry
            with open(data_path, "w") as f:
                json.dump(data, f, indent=2)

    if save_outputs:
        path = save_entropy_results(results, output_dir)
        if verbose:
            print(f"Saved entropy results to: {path}")

    return results

In [ ]:
# Example usage (mirrors the slurm script settings)
results = run_entropy_token_pipeline(
    prompt_idx=1,
    model_name="gemma-2-2b-it",
    device="cuda",
    use_custom_cache=True,
    skip_docstrings=True,
    start_token_offset=0,
    max_tokens_to_analyze=50,
    stop_token_id=1917,
    data_path="../data/external/all_examples_og_prompt_with_position_info.json",
    save_outputs=True,
    output_dir="outputs/entropy/",
    verbose=True,
    use_nodocstring_prompt=True,
    use_pos_info=False,
    pos_info_offset=2,
    top_p_mass=0.9,
    top_max_tokens=200,
    save_full_probs=False,
)

# Access a specific token result
# results["token_results"][token_id]

Starting entropy pipeline for prompt 1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b-it into HookedTransformer
Generating full sequence...
Full generated response: <bos>You are an expert Python programmer, and here is your task: Write a python function to identify non-prime numbers. Your code should pass these tests:

assert is_not_prime(2) == False
assert is_not_prime(10) == True
assert is_not_prime(35) == True
assert is_not_prime(37) == False
Write your code, without docstrings, below starting with "```python" and ending with "```".
```python
def is_not_prime(n):
    if n <= 1:
        return True
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return True
    return False

Generated sequence length: 170 tokens
Analyzing token positions 109 to 159
Saved entropy results to: outputs/entropy/prompt_1/entropy_token_results.json


In [ ]:
# Iterate over prompt_idx from 2 to 99 and run the entropy token pipeline for each
for prompt_idx in range(2, 100):
    print(f"Processing prompt_idx: {prompt_idx}")
    results = run_entropy_token_pipeline(
        prompt_idx=prompt_idx,
        model_name="gemma-2-2b-it",
        device="cuda",
        use_custom_cache=True,
        skip_docstrings=True,
        start_token_offset=0,
        max_tokens_to_analyze=50,
        stop_token_id=1917,
        data_path="../data/external/all_examples_og_prompt_with_position_info.json",
        save_outputs=True,
        output_dir="outputs/entropy/",
        verbose=True,
        use_nodocstring_prompt=True,
        use_pos_info=False,
        pos_info_offset=2,
        top_p_mass=0.9,
        top_max_tokens=200,
        save_full_probs=False,
    )
    # Optionally, access a specific token result for each run
    # e.g., results["token_results"].get(token_id)
